# 🕸️ Notebook 2: Add a New Feature Without Touching the Producer

Start with an e-commerce system. Then add **fraud detection** and **loyalty points** — by writing subscribers only.

## 🛠️ Setup

```bash
cd 05-microservices/event-driven-architecture
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from collections import defaultdict, deque
import threading, time

class EventBus:
    def __init__(self):
        self.q = deque()
        self.subs = defaultdict(list)
        self.alive = True
        threading.Thread(target=self._worker, daemon=True).start()
    def subscribe(self, event, fn):
        self.subs[event].append(fn)
    def publish(self, event, payload):
        self.q.append((event, payload))
    def _worker(self):
        while self.alive:
            if self.q:
                ev, pl = self.q.popleft()
                for fn in list(self.subs[ev]):
                    try: fn(pl)
                    except Exception as e: print('  ⚠️ subscriber error:', e)
            else:
                time.sleep(0.01)

bus = EventBus()

def shipping(p): print('  📦 ship', p['order_id'])
def email(p): print('  ✉️  email', p['email'])
bus.subscribe('order_placed', shipping)
bus.subscribe('order_placed', email)

bus.publish('order_placed', {'order_id':1,'email':'ada@x.io','total':42})
time.sleep(0.2)


Now add two new features without touching the producer:

In [ ]:
def fraud(p):
    if p['total'] > 1000: print('  🚨 fraud alert', p['order_id'])
def loyalty(p):
    points = int(p['total'])
    print(f'  ⭐ +{points} pts for {p["email"]}')

bus.subscribe('order_placed', fraud)
bus.subscribe('order_placed', loyalty)

bus.publish('order_placed', {'order_id':2,'email':'g@x.io','total':1500})
time.sleep(0.2)


### Benefits in practice
- New subscribers = new features with zero producer changes.
- Failures in a subscriber don't break the producer.
- Easier to scale consumers independently.

### Costs
- Flow is implicit — harder to reason about.
- Need schema discipline (events are contracts).
- Delivery semantics (at-least-once, exactly-once) matter.